In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType
from datetime import datetime

SOURCE_BUCKET   = "s3://cms-public-synthetic"
BRONZE_CATALOG  = "cms_beneficiary"
BRONZE_SCHEMA   = "bronze"
BRONZE_FQ       = f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}"
YEARS           = list(range(2015, 2026))  

spark.sql(f"CREATE CATALOG IF NOT EXISTS {BRONZE_CATALOG}")
spark.sql(f"CREATE SCHEMA  IF NOT EXISTS {BRONZE_FQ}")
print("Bronze target:", BRONZE_FQ)

In [0]:
def read_bronze(path: str):
    """Read a CMS beneficiary CSV preserving raw values as strings."""
    return (spark.read
        .option("header", "true")
        .option("delimiter", "|")
        .option("inferSchema", "false")    
        .option("encoding", "ISO-8859-1")   
        .option("mode", "PERMISSIVE")
        .option("columnNameOfCorruptRecord", "_corrupt_record")
        .csv(path))

In [0]:
def with_bronze_meta(df, source_path: str):
    return (df
        .withColumn("_ingested_at", F.lit(datetime.utcnow()).cast("timestamp"))
        .withColumn("_source_file", F.lit(source_path)))

In [0]:
for year in YEARS:
    src = f"{SOURCE_BUCKET}/beneficiary_{year}.csv"
    tgt = f"{BRONZE_FQ}.cms_beneficiary_{year}"

    print(f"→ Reading {src}")
    df = read_bronze(src)
    df = with_bronze_meta(df, src)
    df = df.withColumn("_ref_year", F.lit(year).cast("int"))

    (df.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .partitionBy("_ref_year")
       .saveAsTable(tgt))

    print(f"   ✓ Wrote {tgt}  rows={spark.table(tgt).count():,}")

In [0]:
%sql
CREATE OR REPLACE VIEW cms_beneficiary.bronze.cms_beneficiary_all AS
SELECT * FROM cms_beneficiary.bronze.cms_beneficiary_2015
UNION ALL SELECT * FROM cms_beneficiary.bronze.cms_beneficiary_2016
UNION ALL SELECT * FROM cms_beneficiary.bronze.cms_beneficiary_2017
UNION ALL SELECT * FROM cms_beneficiary.bronze.cms_beneficiary_2018
UNION ALL SELECT * FROM cms_beneficiary.bronze.cms_beneficiary_2019
UNION ALL SELECT * FROM cms_beneficiary.bronze.cms_beneficiary_2020
UNION ALL SELECT * FROM cms_beneficiary.bronze.cms_beneficiary_2021
UNION ALL SELECT * FROM cms_beneficiary.bronze.cms_beneficiary_2022
UNION ALL SELECT * FROM cms_beneficiary.bronze.cms_beneficiary_2023
UNION ALL SELECT * FROM cms_beneficiary.bronze.cms_beneficiary_2024
UNION ALL SELECT * FROM cms_beneficiary.bronze.cms_beneficiary_2025;

In [0]:
def validate_bronze(table: str, expected_min_rows: int = 1):
    df = spark.table(table)
    total = df.count()
    print(f"── {table}")
    print(f"   rows: {total:,}")
    print(f"   cols: {len(df.columns)}")
    print(f"   null BENE_ID: {df.filter(F.col('BENE_ID').isNull()).count():,}")
    print(f"   corrupt recs: "
          f"{df.filter(F.col('_corrupt_record').isNotNull()).count() if '_corrupt_record' in df.columns else 0}")
    assert total >= expected_min_rows, f"{table} has too few rows"
    assert df.filter(F.col("BENE_ID").isNull()).count() == 0, "missing BENE_IDs"

for y in YEARS:
    validate_bronze(f"{BRONZE_FQ}.cms_beneficiary_{y}")

In [0]:
%sql
SELECT _ref_year, COUNT(*) AS rows, COUNT(DISTINCT BENE_ID) AS unique_benes
FROM   cms_beneficiary.bronze.cms_beneficiary_all
GROUP BY _ref_year
ORDER BY _ref_year;

In [0]:
display(spark.sql("SHOW TABLES IN cms_beneficiary.bronze"))
display(spark.sql("SHOW VOLUMES IN cms_beneficiary.bronze"))